# Import and harmonize bike-share data

This is the last step of the workflow in `README.md` (download → extract → **import**). It is cross-platform: it uses project-relative paths and calls the pipeline in `src/data/data.py`.

**Before running:** the ZIPs must be downloaded into `data/archives/` (`python download_data.py`) and extracted into `data/raw/` (`python prepare_data.py`). Start Jupyter from the project root with the virtual environment active.

The notebook reads `data/raw/` and writes the year-partitioned Parquet dataset to `data/processed/trips/` plus a DuckDB database, `bikeshare_dataset.duckdb`.

The input window is capped at trips starting before **2026-09-01**. `Dependent` user-type rows are excluded. `source_file` and `year` are kept for provenance and partitioning.

In [2]:
from pathlib import Path
import logging

import duckdb
import pandas as pd

from src.data.data import build_dataset

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Paths are relative to the project root, so they work on macOS/Linux/Windows.
DATA_DIR = Path('data') / 'raw'
OUTPUT_DIR = Path('data') / 'processed' / 'trips'
DATABASE_PATH = Path('bikeshare_dataset.duckdb')

result = build_dataset(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    database_path=str(DATABASE_PATH),
)

print(f"Total trips: {result['total_trips']:,}")
print('Trips per year:')
for year, count in result['counts_by_year']:
    print(f'  {year}: {count:,}')
print(f"Parquet dataset: {result['output_dir']}")
print(f"DuckDB database: {result['database_path']}")


INFO: Processing historical file: data\raw\Divvy_Stations_Trips_2013\Divvy_Trips_2013.csv
INFO: Processing historical file: data\raw\Divvy_Stations_Trips_2014_Q3Q4\Divvy_Trips_2014-Q3-07.csv
INFO: Processing historical file: data\raw\Divvy_Stations_Trips_2014_Q3Q4\Divvy_Trips_2014-Q3-0809.csv
INFO: Processing historical file: data\raw\Divvy_Stations_Trips_2014_Q3Q4\Divvy_Trips_2014-Q4.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2014_Q1Q2.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015-Q1.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015-Q2.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015_07.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015_08.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015_09.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2015_Q4.csv
INFO: Processing historical file: data\raw\Divvy_Trips_2016_Q1Q2\Divvy_Trips_2016_04.csv
INFO: Processing historical file: data\raw\Divvy

Total trips: 57,299,924
Trips per year:
  2013: 759,788
  2014: 2,454,634
  2015: 3,183,296
  2016: 3,595,293
  2017: 3,828,996
  2018: 3,603,082
  2019: 3,818,004
  2020: 3,541,474
  2021: 5,595,063
  2022: 5,667,717
  2023: 5,719,877
  2024: 5,860,357
  2025: 5,414,328
  2026: 4,258,015
Parquet dataset: data\processed\trips
DuckDB database: bikeshare_dataset.duckdb


## Data checks

The build stops if timestamps, years, or membership categories are invalid. `Dependent` rows are removed during import. Compare the yearly counts above with the totals you expect for your download.

Known limitation: historical station IDs absent from the historical station files are not yet backfilled using 2020 trip coordinates. The cell below shows how many rows per year have missing start or end coordinates; investigate any pre-2020 NULLs before treating the output as final.

In [3]:
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    display(con.sql("""
        SELECT year,
               count(*) AS trips,
               count(*) FILTER (WHERE start_lat IS NULL OR start_lng IS NULL) AS missing_start_coords,
               count(*) FILTER (WHERE end_lat IS NULL OR end_lng IS NULL) AS missing_end_coords
        FROM final_harmonized_trips
        GROUP BY year ORDER BY year
    """).df())

,year,trips,missing_start_coords,missing_end_coords
0,2013,759788,0,0
1,2014,2454634,0,0
2,2015,3183296,0,0
3,2016,3595293,0,0
4,2017,3828996,0,0
5,2018,3603082,26231,26602
6,2019,3818004,121965,123112
7,2020,3541474,0,4254
8,2021,5595063,0,4771
9,2022,5667717,0,5858
